# AquaSense AI — Stage 2: Regression Model Development

**Environment:** Google Colab  
**Dataset:** `aquasense_rainfall_monthly_feature_engineered_reduced.csv` (or the original feature-engineered CSV)  
**Prediction task:** Predict next-month rainfall (`next_month_rainfall_mm`) for a monitoring station.

## Learning objectives

By the end of this notebook, you will be able to:

1. Understand a regression problem.
2. Train a simple Linear Regression model.
3. Train a Polynomial Regression model.
4. Train a Multivariate Linear Regression model.
5. Compare models using MAE, MSE, RMSE and R².
6. Use chronological train, validation and test periods to reduce time leakage.
7. Save the selected model and its preprocessing pipeline.

> **Scope note:** This first model predicts next-month rainfall from historical rainfall features. It does **not** yet predict household water consumption or water scarcity. Those need additional verified datasets such as water consumption, population, reservoir and groundwater measurements.

## 1. Understand the result from Stage 1

Your Stage 1 output contained:

| Statistic | Value |
|---|---:|
| Number of records | 7,228 |
| Mean next-month rainfall | 349.49 mm |
| Standard deviation | 437.34 mm |
| Minimum | 0.20 mm |
| 25th percentile | 94.95 mm |
| Median | 208.00 mm |
| 75th percentile | 407.80 mm |
| Maximum | 5,074.00 mm |

### What does this tell us?

- The mean (349.49) is higher than the median (208). This suggests a **right-skewed distribution**: most station-month rainfall values are lower, while a smaller number are very high.
- The standard deviation (437.34) is large, showing substantial variation.
- The maximum (5,074 mm) is much larger than the 75th percentile (407.8 mm).

**Do not automatically delete the high values.** First check the original source, station, month, units and data quality. Some values could be genuine extremes; others could indicate a recording or aggregation issue. We will evaluate errors in millimetres and inspect residuals.

## 2. Import the required libraries

We use:
- `pandas` to work with the dataset.
- `numpy` for numerical operations.
- `matplotlib` and `seaborn` for charts.
- `scikit-learn` for regression algorithms, preprocessing and evaluation.
- `joblib` to save the final model pipeline.

In [ ]:
import io
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

pd.set_option("display.max_columns", 100)

print("Libraries imported successfully.")

## 3. Upload your feature-engineered CSV

Upload either:
- `aquasense_rainfall_monthly_feature_engineered_reduced.csv`, or
- `aquasense_rainfall_monthly_feature_engineered.csv`.

The reduced file is preferable because it is quicker to upload. Both should contain the target `next_month_rainfall_mm` and a date column such as `year_month`.

You can also upload a `.csv.gz` compressed CSV.

In [ ]:
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file was uploaded. Please upload your AquaSense CSV.")

filename = next(iter(uploaded.keys()))
file_bytes = uploaded[filename]

if filename.lower().endswith(".gz"):
    df = pd.read_csv(io.BytesIO(file_bytes), compression="gzip")
else:
    df = pd.read_csv(io.BytesIO(file_bytes))

print("Loaded file:", filename)
print("Dataset shape:", df.shape)
display(df.head())

## 4. Check the dataset structure

Each row should represent one monitoring station in one month.

We need:
- a station identifier (`station_key`) to identify the monitoring station;
- a month/date (`year_month`) to preserve chronological order;
- the target (`next_month_rainfall_mm`);
- historical rainfall features to use as inputs.

The station identifier and date are used for grouping/splitting, not as direct numerical predictors in this first model.

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("data_type"))

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

## 5. Validate required columns and identify available predictors

The target must be `next_month_rainfall_mm`.

For the three regression experiments:

1. **Simple Linear Regression:** one predictor, preferably `rainfall_lag_1m`.
2. **Polynomial Regression:** the same one predictor transformed to degree 2.
3. **Multivariate Linear Regression:** several numerical rainfall and seasonal predictors.

We intentionally exclude `station_key`, `district`, `year_month`, `next_month_rainfall_mm`, and any other identifiers/target columns from `X`.

In [ ]:
TARGET = "next_month_rainfall_mm"
DATE_COLUMN = "year_month"
STATION_COLUMN = "station_key"

if TARGET not in df.columns:
    raise ValueError(
        f"Target column '{TARGET}' was not found. "
        "Please check that you uploaded the feature-engineered monthly CSV."
    )

if DATE_COLUMN not in df.columns:
    raise ValueError(
        f"Date column '{DATE_COLUMN}' was not found. "
        "This notebook needs the monthly date column to split data chronologically."
    )

# Select the one feature for the simple and polynomial experiments.
single_feature_candidates = [
    "rainfall_lag_1m",
    "rainfall_sum_observed_mm",
    "mean_daily_rainfall_mm"
]
SINGLE_FEATURE = next(
    (c for c in single_feature_candidates if c in df.columns),
    None
)

# Candidate predictors for multivariate linear regression.
multivariate_candidates = [
    "rainfall_sum_observed_mm",
    "mean_daily_rainfall_mm",
    "max_daily_rainfall_mm",
    "rainy_days",
    "heavy_rain_days_20mm",
    "month_sin",
    "month_cos",
    "rainfall_lag_1m",
    "rainfall_prev3m_mean",
    "rainfall_anomaly_vs_prior_climatology"
]
MULTI_FEATURES = [c for c in multivariate_candidates if c in df.columns]

if SINGLE_FEATURE is None:
    raise ValueError(
        "None of the expected single predictor columns were found. "
        "Inspect df.columns and update single_feature_candidates."
    )

if len(MULTI_FEATURES) < 2:
    raise ValueError(
        "Fewer than two multivariate predictor columns were found. "
        "Please check the uploaded CSV columns."
    )

print("Simple / polynomial feature:", SINGLE_FEATURE)
print("Multivariate features:", MULTI_FEATURES)

## 6. Prepare and validate the modelling data

We will:
1. Convert the target and selected features to numeric values.
2. Parse `year_month` as a date.
3. Remove rows where the target or date is missing.
4. Keep the feature values and target in the same row.

Missing values in the input features will be handled by a median imputer **inside each model pipeline**. This is important because the imputer must learn medians from training data only, not from the validation or test data.

In [ ]:
model_columns = list(dict.fromkeys(
    [DATE_COLUMN, TARGET] + ([STATION_COLUMN] if STATION_COLUMN in df.columns else [])
    + [SINGLE_FEATURE] + MULTI_FEATURES
))
work = df[model_columns].copy()

work["_date"] = pd.to_datetime(work[DATE_COLUMN], errors="coerce")
work[TARGET] = pd.to_numeric(work[TARGET], errors="coerce")

for col in set([SINGLE_FEATURE] + MULTI_FEATURES):
    work[col] = pd.to_numeric(work[col], errors="coerce")

# Retain rows with a valid date and known target.
work = work.dropna(subset=["_date", TARGET]).copy()

# Work at monthly frequency for chronological splitting.
work["_period"] = work["_date"].dt.to_period("M")
work = work.sort_values(["_period"] + ([STATION_COLUMN] if STATION_COLUMN in work.columns else []))
work = work.reset_index(drop=True)

print("Rows after date/target validation:", len(work))
print("Unique months:", work["_period"].nunique())
print("Date range:", work["_date"].min().date(), "to", work["_date"].max().date())
print("Target missing after preparation:", work[TARGET].isna().sum())

if work["_period"].nunique() < 12:
    raise ValueError("At least 12 unique months are recommended for chronological modelling.")

display(work.head())

## 7. Inspect the target distribution and high values

Your first histogram showed a long right tail. We will recreate the plot and inspect the largest target values.

A high value is not automatically an error. Check the station and month in the original source before deciding whether to exclude it. For the initial comparison, we keep valid source observations and report errors on the original millimetre scale.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(work[TARGET], bins=40)
axes[0].set_title("Next-month rainfall distribution")
axes[0].set_xlabel("Next-month rainfall (mm)")

axes[1].boxplot(work[TARGET].dropna(), vert=False)
axes[1].set_title("Next-month rainfall boxplot")
axes[1].set_xlabel("Next-month rainfall (mm)")

plt.tight_layout()
plt.show()

print("Target summary:")
display(work[TARGET].describe().to_frame("next_month_rainfall_mm"))

print("Largest 10 target values:")
cols_to_show = ([STATION_COLUMN] if STATION_COLUMN in work.columns else []) + [DATE_COLUMN, TARGET]
display(work.nlargest(10, TARGET)[cols_to_show])

## 8. Define X (independent variables) and y (dependent variable)

In machine learning:
- `X` contains the input features used by a model.
- `y` contains the value the model is learning to predict.

For the first experiment, we use one feature. For the third experiment, we use multiple features.

We are not including the target or station/date identifiers as input features.

In [ ]:
X_simple = work[[SINGLE_FEATURE]].copy()
X_multi = work[MULTI_FEATURES].copy()
y = work[TARGET].copy()

print("Simple X shape:", X_simple.shape)
print("Multivariate X shape:", X_multi.shape)
print("y shape:", y.shape)

print("\nSimple input feature:")
display(X_simple.head())

print("\nMultivariate input features:")
display(X_multi.head())

print("\nTarget:")
display(y.head())

## 9. Split chronologically into training, validation and test periods

For rainfall forecasting, a random split is not appropriate because it can mix earlier and later months.

We use:
- earliest 70% of unique months for training;
- next 15% for validation;
- latest 15% for final testing.

**One-month purge:** The label in a row represents rainfall in the following month. Therefore, a training row whose target month reaches into the validation period must not be used for training. We apply the same rule between validation and test periods. This prevents adjacent forecast labels from crossing into the next evaluation period.

The validation period helps choose a model. The test period is held back for the final, independent evaluation.

In [ ]:
unique_periods = sorted(work["_period"].unique())
n_periods = len(unique_periods)

train_boundary_index = int(n_periods * 0.70)
test_boundary_index = int(n_periods * 0.85)

# Ensure each section has at least one period.
train_boundary_index = max(2, min(train_boundary_index, n_periods - 4))
test_boundary_index = max(train_boundary_index + 2, min(test_boundary_index, n_periods - 2))

validation_start = unique_periods[train_boundary_index]
test_start = unique_periods[test_boundary_index]

# The target month is one month after the feature month.
work["_target_period"] = work["_period"] + 1

train_df = work[work["_target_period"] < validation_start].copy()

validation_df = work[
    (work["_period"] >= validation_start)
    & (work["_target_period"] < test_start)
].copy()

test_df = work[work["_period"] >= test_start].copy()

print("Training period:", train_df["_period"].min(), "to", train_df["_period"].max())
print("Validation period:", validation_df["_period"].min(), "to", validation_df["_period"].max())
print("Test period:", test_df["_period"].min(), "to", test_df["_period"].max())

print("\nTraining rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

if min(len(train_df), len(validation_df), len(test_df)) == 0:
    raise ValueError(
        "A chronological split produced an empty partition. "
        "Check the available date range or reduce the number of split periods."
    )

## 10. Create the X and y partitions

The models will be trained using the training data, compared on validation data, and finally evaluated on test data.

All three experiments use exactly the same time partitions.

In [ ]:
X_train_simple = train_df[[SINGLE_FEATURE]]
X_val_simple = validation_df[[SINGLE_FEATURE]]
X_test_simple = test_df[[SINGLE_FEATURE]]

X_train_multi = train_df[MULTI_FEATURES]
X_val_multi = validation_df[MULTI_FEATURES]
X_test_multi = test_df[MULTI_FEATURES]

y_train = train_df[TARGET]
y_val = validation_df[TARGET]
y_test = test_df[TARGET]

print("Simple regression:", X_train_simple.shape, X_val_simple.shape, X_test_simple.shape)
print("Multivariate regression:", X_train_multi.shape, X_val_multi.shape, X_test_multi.shape)
print("Target partitions:", y_train.shape, y_val.shape, y_test.shape)

## 11. Establish a baseline model

Before training complex models, we create a simple baseline that predicts the **median rainfall in the training set for every validation observation**.

A useful ML model should outperform this baseline. This is a reference point, not one of the three syllabus regression methods.

In [ ]:
baseline = DummyRegressor(strategy="median")
baseline.fit(np.zeros((len(y_train), 1)), y_train)

baseline_val_predictions = baseline.predict(np.zeros((len(y_val), 1)))

def regression_metrics(actual, predicted):
    mse = mean_squared_error(actual, predicted)
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(actual, predicted)
    }

baseline_metrics = regression_metrics(y_val, baseline_val_predictions)
print("Baseline validation metrics:")
display(pd.DataFrame([baseline_metrics], index=["Median baseline"]).round(3))

## 12. Model 1 — Simple Linear Regression

Simple Linear Regression uses **one independent variable** to predict one dependent variable.

Conceptually, it fits a straight line:

`y = b0 + b1*x`

- `x` is the selected rainfall feature.
- `b0` is the intercept.
- `b1` is the slope.
- `y` is predicted next-month rainfall.

We use a median imputer inside the pipeline so missing input values are handled using training data only.

In [ ]:
simple_linear_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("regressor", LinearRegression())
])

simple_linear_model.fit(X_train_simple, y_train)
simple_val_predictions = simple_linear_model.predict(X_val_simple)

print("Simple Linear Regression validation metrics:")
display(pd.DataFrame(
    [regression_metrics(y_val, simple_val_predictions)],
    index=["Simple Linear Regression"]
).round(3))

## 13. Model 2 — Polynomial Regression

Polynomial Regression can model a curved relationship.

For degree 2, the single input `x` is expanded to include:
- `x`
- `x²`

A linear regression model is then fitted on these transformed features. The relationship is curved in the original input space, although the model is linear in its coefficients.

We start with degree 2 to avoid introducing unnecessary complexity.

In [ ]:
polynomial_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

polynomial_model.fit(X_train_simple, y_train)
polynomial_val_predictions = polynomial_model.predict(X_val_simple)

print("Polynomial Regression (degree 2) validation metrics:")
display(pd.DataFrame(
    [regression_metrics(y_val, polynomial_val_predictions)],
    index=["Polynomial Regression"]
).round(3))

## 14. Model 3 — Multivariate Linear Regression

Multivariate Linear Regression uses **more than one independent variable** to predict one numerical target.

Conceptually:

`y = b0 + b1*x1 + b2*x2 + ... + bn*xn`

Each `x` represents a different rainfall or seasonal feature.

This experiment uses the available selected numerical features. The imputer and scaler are fitted on training data only.

In [ ]:
multivariate_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

multivariate_model.fit(X_train_multi, y_train)
multivariate_val_predictions = multivariate_model.predict(X_val_multi)

print("Multivariate Linear Regression validation metrics:")
display(pd.DataFrame(
    [regression_metrics(y_val, multivariate_val_predictions)],
    index=["Multivariate Linear Regression"]
).round(3))

## 15. Compare the models on validation data

We use identical validation observations for all models.

Metric interpretation:
- **MAE:** average absolute prediction error in millimetres. Lower is better.
- **MSE:** average squared error. Lower is better; large errors receive stronger penalties.
- **RMSE:** square root of MSE, in millimetres. Lower is better.
- **R²:** proportion of target variation explained relative to a mean baseline. Closer to 1 is generally better; it can be negative.

We choose the model using validation RMSE, while also examining MAE and R². Do not assume accuracy is the right metric for regression.

In [ ]:
validation_predictions = {
    "Simple Linear Regression": simple_val_predictions,
    "Polynomial Regression (degree 2)": polynomial_val_predictions,
    "Multivariate Linear Regression": multivariate_val_predictions
}

validation_rows = []
for model_name, predictions in validation_predictions.items():
    row = {"Model": model_name}
    row.update(regression_metrics(y_val, predictions))
    validation_rows.append(row)

validation_results = pd.DataFrame(validation_rows).sort_values("RMSE")
display(validation_results.round(3))

best_model_name = validation_results.iloc[0]["Model"]
print("Best model selected using validation RMSE:", best_model_name)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(validation_results["Model"], validation_results["RMSE"])
axes[0].set_title("Validation RMSE (lower is better)")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(validation_results["Model"], validation_results["MAE"])
axes[1].set_title("Validation MAE (lower is better)")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## 16. Final evaluation on the unseen test period

We now refit each model on the combined training and validation partitions, then evaluate it once on the held-out test period.

The test period was not used to select the model. This gives a more honest estimate of performance on later months.

In [ ]:
train_validation_df = pd.concat([train_df, validation_df]).sort_values(
    ["_period"] + ([STATION_COLUMN] if STATION_COLUMN in work.columns else [])
)

X_trainval_simple = train_validation_df[[SINGLE_FEATURE]]
X_test_simple_final = test_df[[SINGLE_FEATURE]]

X_trainval_multi = train_validation_df[MULTI_FEATURES]
X_test_multi_final = test_df[MULTI_FEATURES]

y_trainval = train_validation_df[TARGET]
y_test_final = test_df[TARGET]

final_models = {
    "Simple Linear Regression": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("regressor", LinearRegression())
    ]),
    "Polynomial Regression (degree 2)": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("regressor", LinearRegression())
    ]),
    "Multivariate Linear Regression": Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("regressor", LinearRegression())
    ])
}

test_predictions = {}
test_rows = []

for name, model in final_models.items():
    if name == "Simple Linear Regression":
        X_fit, X_evaluate = X_trainval_simple, X_test_simple_final
    elif name == "Polynomial Regression (degree 2)":
        X_fit, X_evaluate = X_trainval_simple, X_test_simple_final
    else:
        X_fit, X_evaluate = X_trainval_multi, X_test_multi_final

    model.fit(X_fit, y_trainval)
    predictions = model.predict(X_evaluate)
    test_predictions[name] = predictions

    row = {"Model": name}
    row.update(regression_metrics(y_test_final, predictions))
    test_rows.append(row)

test_results = pd.DataFrame(test_rows).sort_values("RMSE")
display(test_results.round(3))

## 17. Plot actual vs predicted values for the validation-selected model

The selected model is the model that had the lowest validation RMSE. We now view its predictions on the test data.

Points closer to the diagonal line represent more accurate predictions.

In [ ]:
selected_model = final_models[best_model_name]

if best_model_name in [
    "Simple Linear Regression",
    "Polynomial Regression (degree 2)"
]:
    selected_test_predictions = test_predictions[best_model_name]
else:
    selected_test_predictions = test_predictions[best_model_name]

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test_final, selected_test_predictions, alpha=0.45)

plot_min = min(y_test_final.min(), selected_test_predictions.min())
plot_max = max(y_test_final.max(), selected_test_predictions.max())
ax.plot([plot_min, plot_max], [plot_min, plot_max], linestyle="--")

ax.set_xlabel("Actual next-month rainfall (mm)")
ax.set_ylabel("Predicted next-month rainfall (mm)")
ax.set_title(f"Actual vs Predicted — {best_model_name}")
plt.tight_layout()
plt.show()

## 18. Residual analysis

A residual is:

`Residual = Actual value - Predicted value`

- A positive residual means the model predicted too little.
- A negative residual means the model predicted too much.
- A good model should not show a strong systematic pattern in its residuals.

Rainfall has a right-skewed distribution, so pay special attention to large errors during high-rainfall months.

In [ ]:
residuals = y_test_final.to_numpy() - selected_test_predictions

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(selected_test_predictions, residuals, alpha=0.45)
axes[0].axhline(0, linestyle="--")
axes[0].set_xlabel("Predicted rainfall (mm)")
axes[0].set_ylabel("Residual (mm)")
axes[0].set_title("Residuals vs predicted")

axes[1].hist(residuals, bins=35)
axes[1].set_title("Residual distribution")
axes[1].set_xlabel("Residual (mm)")

plt.tight_layout()
plt.show()

print("Mean residual:", round(float(np.mean(residuals)), 3))
print("Median absolute residual:", round(float(np.median(np.abs(residuals))), 3))

## 19. Save the trained regression pipelines and result tables

We save:
- the three fitted regression pipelines;
- the validation comparison;
- the final test comparison;
- the selected model name.

The pipeline includes preprocessing, so the same imputation/scaling/feature transformation is applied when the model is later used in the AquaSense application.

In [ ]:
os.makedirs("aquasense_regression_artifacts", exist_ok=True)

for name, model in final_models.items():
    safe_name = (
        name.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace(".", "")
    )
    joblib.dump(
        model,
        f"aquasense_regression_artifacts/{safe_name}.joblib"
    )

validation_results.to_csv(
    "aquasense_regression_artifacts/validation_model_comparison.csv",
    index=False
)
test_results.to_csv(
    "aquasense_regression_artifacts/test_model_comparison.csv",
    index=False
)

with open("aquasense_regression_artifacts/selected_model.txt", "w") as f:
    f.write(best_model_name)

print("Saved all regression pipelines and comparison results.")

## 20. Download the model artifacts

Download the folder as a ZIP file so you can keep it for the later Gradio application.

In [ ]:
import shutil

shutil.make_archive(
    "aquasense_regression_artifacts",
    "zip",
    "aquasense_regression_artifacts"
)

files.download("aquasense_regression_artifacts.zip")

## 21. Conclusion — fill this in using your actual results

After executing the notebook, complete the following points using the generated tables:

1. The dataset contained ______ usable station-month records.
2. The selected single input feature was ______.
3. The multivariate model used ______ features.
4. The model with the lowest validation RMSE was ______.
5. Its final test MAE was ______ mm.
6. Its final test RMSE was ______ mm.
7. Its final test R² was ______.
8. The main limitation observed in the results was ______.

### Interpretation and limitations

- The target distribution is right-skewed, so a few very high rainfall values can strongly affect MSE and RMSE.
- Results are specific to the stations, dates and features available in this dataset.
- A model that performs well on rainfall data is not automatically a good model for water consumption or scarcity.
- Further improvement may include more verified stations, additional weather variables, careful feature engineering, and testing other regression algorithms.

## 22. What comes next?

After you execute this notebook and share the validation/test comparison tables, we will:
1. Interpret the results together.
2. Identify underfitting or overfitting signs.
3. Decide whether a nonlinear or transformed-target experiment is justified.
4. Proceed to the classification stage using a defensible rainfall-category target.